In [ ]:
# Ubuntu 24.04 system libraries required by headless Chromium.
!sudo apt-get update -y
!sudo apt-get install -y libnss3 libnspr4 libatk1.0-0t64 libatk-bridge2.0-0t64 libatspi2.0-0t64 libcups2t64 libasound2t64 libgtk-3-0t64 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libdrm2 libx11-xcb1 libxcb1 libxcb-dri3-0 libxshmfence1 libpango-1.0-0 libcairo2 libxrender1 libfontconfig1 libfreetype6 libglib2.0-0t64 libdbus-1-3 libexpat1 libxext6 libx11-6 libegl1 fonts-liberation


# USMAN AND UMAR — Colab Backend with zrok

This notebook runs the repository's existing FastAPI OpenSwarm bridge and Browser Use helper in Google Colab. It does not rewrite either implementation. The API is exposed through a stable public zrok URL; the FastAPI endpoint still requires `Authorization: Bearer <APP_CLIENT_TOKEN>`.

**zrok CLI compatibility:** Current zrok is v2.0. The old `zrok reserve` and `zrok share reserved` commands were removed. This notebook uses the v2 equivalent: create a persistent public name with `zrok2 create name`, then attach it with `zrok2 share public ... -n public:<name>`. The requested Secret names are retained: `ZROK_ENABLE_TOKEN` contains the account token; `ZROK_SHARE_TOKEN` contains the public name (for example `usman-umar`), not a credential/token.

## One-time account and Colab setup

1. Create/sign in to a free account at [myzrok.io](https://myzrok.io/). Open the zrok/API console and its **Get Started** wizard, then copy the account token. Treat it as a password.
2. Choose a public name, such as `usman-umar`. When the reservation cell below runs the first time, it reserves `public:<name>` for your account. The expected URL is `https://<name>.share.zrok.io` (for example `https://usman-umar.share.zrok.io`). You do not generate a separate public share token in zrok v2.
3. In Colab, open the **Secrets** panel using the key icon in the left sidebar. Add the following names and values, and enable notebook access for each secret:
   - `ZROK_ENABLE_TOKEN` — the zrok account token from the wizard.
   - `ZROK_SHARE_TOKEN` — your chosen public name, e.g. `usman-umar`.
   - `APP_CLIENT_TOKEN` — a strong random Bearer secret; save the same value in the Android app's Cloud Backend settings.
   - `OPEN_SWARM_EXECUTE_URL` — the real upstream OpenSwarm HTTPS execution URL, including its path.
   - `OPEN_SWARM_BEARER_TOKEN` — only if the upstream OpenSwarm endpoint requires its own Bearer token.
   - `OPENROUTER_API_KEY` — optional; used by the existing interactive Browser Use helper. If absent, that helper falls back to local Ollama.
4. Run the notebook cells from top to bottom. Keep this Colab runtime alive while the public URL is needed. The URL is public, but calls to `/execute` still require the app Bearer token.

The hosted free tier currently lists 5 GB/day, 25 environments, and 50 share backends, with no credit card required. Free public shares may show an interstitial page to visitors; the runtime must stay connected for the tunnel to work.


## 1. Install the pinned Linux zrok2 client

This installs the official x86-64 or ARM64 binary release without adding `pyngrok`.


In [ ]:
import io
import os
import platform
import shutil
import subprocess
import tarfile
import urllib.request
from pathlib import Path

ZROK_VERSION = "2.0.8"
arch_map = {"x86_64": "amd64", "aarch64": "arm64", "arm64": "arm64"}
zrok_arch = arch_map.get(platform.machine().lower())
if not zrok_arch:
    raise RuntimeError(f"Unsupported Colab CPU architecture: {platform.machine()}")

archive_url = (
    f"https://github.com/openziti/zrok/releases/download/v{ZROK_VERSION}/"
    f"zrok_{ZROK_VERSION}_linux_{zrok_arch}.tar.gz"
)
with urllib.request.urlopen(archive_url, timeout=120) as response:
    archive_bytes = response.read()

with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
    binary_member = next(
        (member for member in archive.getmembers() if Path(member.name).name == "zrok2"),
        None,
    )
    if binary_member is None:
        raise RuntimeError("The pinned zrok release archive did not contain zrok2.")
    binary_source = archive.extractfile(binary_member)
    if binary_source is None:
        raise RuntimeError("Could not read zrok2 from the release archive.")
    binary_bytes = binary_source.read()

install_dir = Path("/usr/local/bin")
try:
    install_dir.mkdir(parents=True, exist_ok=True)
    binary_path = install_dir / "zrok2"
    binary_path.write_bytes(binary_bytes)
    binary_path.chmod(0o755)
except PermissionError:
    install_dir = Path("/content/bin")
    install_dir.mkdir(parents=True, exist_ok=True)
    binary_path = install_dir / "zrok2"
    binary_path.write_bytes(binary_bytes)
    binary_path.chmod(0o755)
    os.environ["PATH"] = f"{install_dir}:{os.environ.get('PATH', '')}"

version = subprocess.run([str(binary_path), "version"], check=True, text=True, capture_output=True)
print(version.stdout.strip())


## 2. Load the unchanged backend and Browser Use helper

The notebook clones this branch's `cloud-backend/` folder, installs its FastAPI dependencies, and installs Browser Use/Chromium for the existing read-only helper.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/saedhawwa23-rgb/usman-and-umar-app.git"
BRANCH = "feature/hybrid-task-orchestrator"
PROJECT_DIR = Path("/content/usman-and-umar-app")

if not (PROJECT_DIR / ".git").exists():
    if PROJECT_DIR.exists():
        raise RuntimeError(
            f"{PROJECT_DIR} exists but is not a Git checkout. Move it aside and rerun this cell."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

BACKEND_DIR = PROJECT_DIR / "cloud-backend"
if not (BACKEND_DIR / "main.py").is_file() or not (BACKEND_DIR / "browser_use_interactive.py").is_file():
    raise RuntimeError("The expected FastAPI or Browser Use source file is missing from the checkout.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND_DIR / "requirements.txt"), "browser-use"],
    check=True,
)
PLAYWRIGHT_OS_PACKAGES = [
    "libnss3", "libnspr4", "libatk1.0-0t64", "libatk-bridge2.0-0t64",
    "libatspi2.0-0t64", "libcups2t64", "libasound2t64", "libgtk-3-0t64",
    "libxkbcommon0", "libxcomposite1", "libxdamage1", "libxfixes3",
    "libxrandr2", "libgbm1", "libdrm2", "libx11-xcb1", "libxcb1",
    "libxcb-dri3-0", "libxshmfence1", "libpango-1.0-0", "libcairo2",
    "libxrender1", "libfontconfig1", "libfreetype6", "libglib2.0-0t64",
    "libdbus-1-3", "libexpat1", "libxext6", "libx11-6", "libegl1",
    "fonts-liberation",
]
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = "/content/.cache/ms-playwright"
os.makedirs(os.environ["PLAYWRIGHT_BROWSERS_PATH"], exist_ok=True)
os.environ["PLAYWRIGHT_SKIP_VALIDATE_HOST_REQUIREMENTS"] = "1"
playwright_install_command = [sys.executable, "-m", "playwright", "install", "chromium"]
try:
    subprocess.run(playwright_install_command, check=True)
except subprocess.CalledProcessError:
    print("Playwright install failed; installing Chromium system libraries and retrying once.")
    subprocess.run(["sudo", "apt-get", "update", "-y"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "-y", *PLAYWRIGHT_OS_PACKAGES], check=True)
    subprocess.run(playwright_install_command, check=True)
print("Backend and Browser Use dependencies are ready.")


## 3. Read Colab Secrets (values are never printed)


In [ ]:
import os
from google.colab import userdata

def read_secret(name, *, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    value = (value or "").strip()
    if required and not value:
        raise RuntimeError(
            f"Missing Colab Secret {name}. Add it in the Secrets panel and enable notebook access."
        )
    return value

ZROK_ENABLE_TOKEN = read_secret("ZROK_ENABLE_TOKEN", required=True)
ZROK_SHARE_TOKEN = read_secret("ZROK_SHARE_TOKEN", required=True).lower()
APP_CLIENT_TOKEN = read_secret("APP_CLIENT_TOKEN", required=True)
OPEN_SWARM_EXECUTE_URL = read_secret("OPEN_SWARM_EXECUTE_URL")
OPEN_SWARM_BEARER_TOKEN = read_secret("OPEN_SWARM_BEARER_TOKEN")
OPENROUTER_API_KEY = read_secret("OPENROUTER_API_KEY")

if not (4 <= len(ZROK_SHARE_TOKEN) <= 32) or any(
    not (char.isascii() and (char.islower() or char.isdigit() or char == "-"))
    for char in ZROK_SHARE_TOKEN
):
    raise ValueError("ZROK_SHARE_TOKEN must be a 4–32 character lowercase public name (letters, digits, hyphens).")

os.environ["APP_CLIENT_TOKEN"] = APP_CLIENT_TOKEN
os.environ["OPEN_SWARM_EXECUTE_URL"] = OPEN_SWARM_EXECUTE_URL
os.environ["OPEN_SWARM_BEARER_TOKEN"] = OPEN_SWARM_BEARER_TOKEN
os.environ["ENABLE_MOCK_EXECUTION"] = "false"
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

if not OPEN_SWARM_EXECUTE_URL:
    print("Warning: OPEN_SWARM_EXECUTE_URL is unset; health works, but real /execute requests will return a configuration error.")
print("Secrets loaded into this runtime; secret values are hidden.")


## 4. Enable zrok2 and reserve the stable public name

Run this after setting the Colab Secrets. The name creation is safe to rerun if the name is already reserved in this zrok account.


In [ ]:
import subprocess

status = subprocess.run(["zrok2", "status"], capture_output=True, text=True)
environment_ready = (
    status.returncode == 0
    and "EnvZId" in status.stdout
    and "<<SET>>" in status.stdout
)
if not environment_ready:
    subprocess.run(
        ["zrok2", "enable", ZROK_ENABLE_TOKEN],
        check=True,
        capture_output=True,
        text=True,
    )
print("zrok2 environment is enabled.")

reserve = subprocess.run(
    ["zrok2", "create", "name", "-n", "public", ZROK_SHARE_TOKEN],
    capture_output=True,
    text=True,
)
if reserve.returncode == 0:
    print(f"Reserved public name: public:{ZROK_SHARE_TOKEN}")
else:
    error_text = (reserve.stdout + "\n" + reserve.stderr).strip()
    lower_error = error_text.lower()
    if any(marker in lower_error for marker in ("already exists", "already reserved", "name conflict", "409")):
        print(f"Public name public:{ZROK_SHARE_TOKEN} already exists; reusing it.")
    else:
        raise RuntimeError(f"Could not reserve the zrok public name (zrok2 exit {reserve.returncode}): {error_text[-1200:]}")


## 5. Start the FastAPI server on port 7860

This launches the existing `cloud-backend/main.py` with its original auth and upstream behavior. Do not turn on mock mode for real tasks.


In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

API_BASE = "http://127.0.0.1:7860"

def local_health_ok():
    try:
        with urllib.request.urlopen(f"{API_BASE}/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if local_health_ok():
    print("FastAPI is already healthy on port 7860.")
else:
    server_log_path = Path("/tmp/usman-fastapi.log")
    server_log = open(server_log_path, "a", encoding="utf-8")
    fastapi_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"],
        cwd=BACKEND_DIR,
        env=os.environ.copy(),
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(30):
        if local_health_ok():
            break
        if fastapi_process.poll() is not None:
            break
        time.sleep(1)
    if not local_health_ok():
        server_log.flush()
        server_log.close()
        log_text = server_log_path.read_text(encoding="utf-8", errors="replace")[-3000:]
        raise RuntimeError(f"FastAPI did not become healthy. Recent server log:\n{log_text}")
    print("FastAPI health check passed at /health.")


## 6. Publish the API at the reserved zrok URL

This is the v2 replacement for `zrok share reserved`. Keep the cell's zrok process running while the backend is in use.


In [ ]:
import re
import subprocess
import time
from pathlib import Path

expected_public_url = f"https://{ZROK_SHARE_TOKEN}.share.zrok.io"
share_log_path = Path("/tmp/usman-zrok-share.log")

if "zrok_share_process" in globals() and zrok_share_process.poll() is None:
    print(f"A zrok share is already running. Expected stable URL: {expected_public_url}")
else:
    share_log = open(share_log_path, "a", encoding="utf-8")
    zrok_share_process = subprocess.Popen(
        [
            "zrok2", "share", "public", "http://127.0.0.1:7860",
            "-n", f"public:{ZROK_SHARE_TOKEN}",
            "--headless", "--force-local",
        ],
        env=os.environ.copy(),
        stdout=share_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(30):
        if zrok_share_process.poll() is not None:
            break
        time.sleep(1)
    share_log.flush()
    log_text = share_log_path.read_text(encoding="utf-8", errors="replace")[-5000:]
    if zrok_share_process.poll() is not None:
        share_log.close()
        raise RuntimeError(f"zrok2 share exited with code {zrok_share_process.returncode}. Recent log:\n{log_text}")
    found_url = re.search(r"https://[A-Za-z0-9.-]+\.share\.zrok\.io", log_text)
    print("zrok2 share process is running.")
    print(f"Expected stable URL: {found_url.group(0) if found_url else expected_public_url}")
    if not found_url:
        print(f"If zrok has not printed the URL yet, inspect {share_log_path} after a few seconds.")
    print("Set the Android app's Cloud Backend URL to this base URL; the app appends /execute.")


## 7. Optional: use the unchanged interactive Browser Use helper

The helper restricts browsing to the hostnames you explicitly allow and runs read-only tasks. `OPENROUTER_API_KEY` is optional and is read from Colab Secrets; without it, the helper uses local Ollama.


In [ ]:
import sys
sys.path.insert(0, str(BACKEND_DIR))
from browser_use_interactive import run_browser_task

result = await run_browser_task(
    "Read the Python installation instructions and summarize the requirements.",
    allowed_domains=["docs.python.org"],
    max_steps=8,
)
print(result)
